# N169 · 从暴力到最优的优化阶梯

**目标：** 在每次优化后保留独立可验证的参照。

**先修：** N015, N032, N038, N135, N056。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 重复计算；状态压缩；单调候选；分治统计；操作数；对拍。

## 从问题到算法

优化前先固定计数对象：每个区间对应一对不同位置的前缀 `(i,j)`，i<j。重复的前缀数值不能去重，因为它们代表不同区间。

直接累加枚举是 O(n²)。改成前缀和只能省掉每个区间的求和，还没有减少区间对数。真正的改进是按原下标分治：左右半边内部递归统计，跨半边的所有左前缀都天然早于右前缀；再按数值排序，用两个单向边界计数。

左前缀 p 增大时，右前缀合法区间 `[p+lower,p+upper]` 也右移，所以两个边界都不用回退。演示统计实际比较次数，不用一次毫秒曲线冒充复杂度证明。

## 最小实现

**本章接口：** `count_range_sum_bruteforce(nums, lower, upper)`、`count_range_sum_merge(nums, lower, upper)`、`benchmark_operation_counts()`

In [1]:
def count_range_sum_bruteforce(nums, lower, upper, stats=None):
    if lower>upper:
        raise ValueError('lower must not exceed upper')
    answer = 0
    if stats is not None:
        stats['extensions']=0
    for left in range(len(nums)):
        total=0
        for right in range(left,len(nums)):
            total+=nums[right]
            answer += lower<=total<=upper
            if stats is not None:stats['extensions']+=1
    return answer

def count_range_sum_merge(nums, lower, upper, stats=None):
    if lower>upper:
        raise ValueError('lower must not exceed upper')
    prefix=[0]
    for value in nums:prefix.append(prefix[-1]+value)
    operations=0
    def sort_count(values):
        nonlocal operations
        if len(values)<=1:return values,0
        middle=len(values)//2
        left,a=sort_count(values[:middle])
        right,b=sort_count(values[middle:])
        answer=a+b
        lo=hi=0
        for value in left:
            while lo<len(right):
                operations+=1
                if right[lo]-value>=lower:break
                lo+=1
            while hi<len(right):
                operations+=1
                if right[hi]-value>upper:break
                hi+=1
            answer+=hi-lo
        merged=[];i=j=0
        while i<len(left) and j<len(right):
            operations+=1
            if left[i]<=right[j]:merged.append(left[i]);i+=1
            else:merged.append(right[j]);j+=1
        merged.extend(left[i:]);merged.extend(right[j:])
        return merged,answer
    answer=sort_count(prefix)[1]
    if stats is not None:stats['comparisons']=operations
    return answer

def benchmark_operation_counts():
    rows=[]
    for n in [16,32,64,128,256]:
        nums=[(i*17)%11-5 for i in range(n)]
        slow={};fast={}
        a=count_range_sum_bruteforce(nums,-3,4,slow)
        b=count_range_sum_merge(nums,-3,4,fast)
        assert a==b
        rows.append((n,slow['extensions'],fast['comparisons']))
    return rows

## 正确性、前提与复杂度

每一对前缀恰在最小的、将二者分入不同半边的递归节点被统计一次。右边界条件 `<lower` 和 `<=upper` 分别对应闭区间两端；重复值保留位置重数。分治合并没有遗漏或重复。

**代价：** 递推 T(n)=2T(n/2)+O(n)，时间 O(n log n)，本实现峰值辅助空间 O(n)，递归深度 O(log n)。Python 大整数算术成本还取决于整数位长；这里的 n 复杂度采用字长算术模型。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['n','brute extensions','merge comparisons'],benchmark_operation_counts())
show_table(['nums','range','count'],[([-2,5,-1],[-2,2],count_range_sum_merge([-2,5,-1],-2,2)),([0,0],[0,0],count_range_sum_merge([0,0],0,0))])

n,brute extensions,merge comparisons
16,136,141
32,528,359
64,2080,858
128,8256,2042
256,32896,4735


nums,range,count
"[-2, 5, -1]","[-2, 2]",3
"[0, 0]","[0, 0]",3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
import itertools,math
assert count_range_sum_merge([-2,5,-1],-2,2)==3
assert count_range_sum_merge([0,0,0],0,0)==6
for n in range(6):
    for nums in itertools.product([-1,0,1],repeat=n):
        for lower,upper in [(-2,2),(0,0),(-1,0),(1,2)]:
            assert count_range_sum_merge(nums,lower,upper)==count_range_sum_bruteforce(nums,lower,upper)
for n,brute_ops,merge_ops in benchmark_operation_counts():
    assert brute_ops==n*(n+1)//2
    assert merge_ops<=10*(n+1)*math.ceil(math.log2(n+1))
assert count_range_sum_merge([10**30,-10**30],0,0)==1
print('N169: 所有本章断言通过')

N169: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 为范围和同时实现Fenwick替代解。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 缩减对拍失败样本到最小反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 327. Count of Range Sum（canonical）
- 239. Sliding Window Maximum（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。